# Stream-CQSA v3 on CITA (H100): correctness and the E0 / E1 / E3 / E4 / E6 experiments

This notebook is the one artefact to bring back. Every cell prints a header, catches its own errors (the
traceback stays in the cell output) and never stops the cells after it. `python run_cita.py` executes it with
nbclient and saves the outputs into `run_cita_executed.ipynb` after every cell; the last cell (and
`run_cita.py` once more, after the final save) packs `results/v3_cita/`, the executed notebook, the build logs
and the doctor output into `cita_results_<host>_<date>.tar.gz`.

**Before running**: `bash setup_cita.sh` in the activated venv (see README.md). The series that run are the
ones the install provides: `classic_Triton` and `wave_Triton` always; `classic_CUDA` when `cqsa_cuda` was
built (tier b); `wave_CUDA` when `cqsa_native` was built (tier c). The baselines are torch SDPA on its flash
backend (`SDPA`) and, only when `flash_attn` is installed, FlashAttention-2 (`FA-2`). **Without `flash_attn`
the reference arm of every error column is SDPA-flash** (the harness records it as `ref_arm` in every row and
the report labels it so; it is never called FA-2).

Every Stream-CQSA row pins `itr` and `c` and asserts the decomposition it reports (`tests/_decomp.py`); a row
whose engine took the monolithic path is an error row, not a measurement.

## Knobs (environment variables)

| variable | default | meaning |
|---|---|---|
| `CITA_QUICK=1` | off | the ~20-minute first pass: pytest + inline correctness, E0 at a reduced grid (`c` in {7, 31}, `itr` 1), E1 forward up to N = 1M; E1 fwd+bwd, E3, E4, E6 are skipped |
| `CITA_MAX_N` | 4194304 | largest N of the E1 ladder (set 2097152 on an OOM / to save time) |
| `CITA_BUDGET_SCALE` | 1 | multiplies every cell's time budget (`--budget-s`); rows that would not fit are deferred, never killed, and a later `python run_cita.py` on the same results directory runs only the missing rows |
| `CITA_FULL=1` | off | no time budgets at all (expect several hours; see the table below) |
| `CITA_SERIES` | auto | comma list of series to run instead of the detected ones (e.g. `classic_Triton,wave_Triton`) |
| `CITA_OUT` | `results/v3_cita` | the results directory (`results.jsonl` is append-only and resumable) |
| `CITA_LABEL` | empty | a free-text label stored in `env.json` (the Della plumbing check sets "Della vis1 plumbing check") |
| `CITA_SMOKE=1` | off | plumbing only: N = 16K / 32K, one run per row, no budgets. Never a measurement. |
| `CUDA_VISIBLE_DEVICES` | `0` | one GPU; the notebook sets it to 0 when unset |

## Wall-time reference (A100-SXM4-80GB, `results/v3/report.md`, jobs 15966097 / 16003186)

The H100 will differ (no factor is assumed here: the kernels are memory- and tensor-core-bound in different
proportions, and the Triton kernels compile differently per architecture). Rows run cheapest first inside a
cell, so a budget cuts the largest configurations, not the small ones.

| cell | rows | A100 wall (whole cell, incl. references) | default budget (x `CITA_BUDGET_SCALE`) |
|---|---|---|---|
| pytest (4 files) | 30 GPU rows + CPU tests | ~2-4 min (measured on vis1 with the CUDA module) | none |
| inline correctness | 2 N x every series | ~1 min | none |
| E0 (128 rows, reps 1, ref FA-2/SDPA-flash) | 128 | 21 min measured | 30 min (quick: 15 min on 48 rows) |
| E1 forward (36 rows, reps 1 measured; reps 2 here) | 36 | 32 min measured at reps 1; ~45 min at reps 2 | 30 min |
| E1 forward+backward (36 rows, reps 2) | 36 | not measured on the A100; ~3.6x the forward: ~2.5 h estimated | 30 min |
| E3 forward (36 rows: 9 c x 4 series, reps 3 measured; reps 2 here) | 36 | 25 min measured at reps 3 | 15 min |
| E4 (32 rows, reps 3 measured; reps 2 here) | 32 | 45 min measured (24 of 32 rows) | 15 min |
| E6 (48 rows, reps 2) | 48 | not measured; ~60-75 min estimated | 15 min |
| report + tarball | - | < 1 min | none |

With the default budgets the whole notebook is bounded by ~2 h 15 min of experiments plus ~10 min of
tests; `CITA_QUICK=1` by ~20 min. Re-running `python run_cita.py` continues where the budgets stopped.

In [ ]:
# ---- helpers shared by every cell (no GPU work in this cell) ------------------------------------------------
import os, sys, json, subprocess, time, socket, traceback, shutil, glob, io, tarfile, base64

HERE = os.getcwd()                                   # run_cita.py / nbconvert run the notebook from the bundle directory
SRC = os.path.join(HERE, "src")
BENCH = os.path.join(SRC, "experiments", "v3", "bench.py")
PY = sys.executable
K, M = 1024, 1 << 20
N_LADDER = [128 * K, 256 * K, 512 * K, M, 2 * M, 4 * M]
QUICK = os.environ.get("CITA_QUICK", "") == "1"
SMOKE = os.environ.get("CITA_SMOKE", "") == "1"           # plumbing check: tiny N, never a measurement
FULL = os.environ.get("CITA_FULL", "") == "1"             # no per-cell time budgets
MAX_N = int(os.environ.get("CITA_MAX_N", 4 * M))
SCALE = float(os.environ.get("CITA_BUDGET_SCALE", "1"))
LABEL = os.environ.get("CITA_LABEL", "")
OUT = os.environ.get("CITA_OUT") or os.path.join(HERE, "results", "v3_cita" + ("_smoke" if SMOKE else ""))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")       # one GPU for every subprocess
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.makedirs(os.path.join(OUT, "logs"), exist_ok=True)
STATUS = {}                                               # cell -> ok / note (cell_status.json; the last cell prints it)
T_START = time.time()
SERIES = ["classic_Triton", "wave_Triton"]                # refined by the environment cell from kernels_available()
BASELINES = ["SDPA"]
assert os.path.exists(BENCH), f"{BENCH} not found: run this notebook from the bundle directory (python run_cita.py)"


def header(name, note=""):
    print("=" * 110)
    print(f"## {name}   [{time.strftime('%Y-%m-%d %H:%M:%S')}  +{(time.time() - T_START) / 60:.1f} min]" + (f"   {note}" if note else ""))
    print("=" * 110, flush=True)


def record(name, ok, note=""):
    STATUS[name] = dict(ok=bool(ok), note=str(note)[:300], at_min=round((time.time() - T_START) / 60, 1))
    with open(os.path.join(OUT, "cell_status.json"), "w") as f:
        json.dump(STATUS, f, indent=1)
    print(f"-> {name}: {'OK' if ok else 'FAILED'}" + (f" ({note})" if note else ""), flush=True)


def run(cmd, log, cwd=None, env=None):
    """Run cmd, streaming its output into this cell and into OUT/logs/<log>; returns the exit code (never raises)."""
    path = os.path.join(OUT, "logs", log)
    print("$ " + " ".join(str(c) for c in cmd), flush=True)
    t0 = time.time()
    # attention(kernel="triton") sets CQSA_FORWARD / CQSA_BACKWARD=triton in the calling process (the inline cell) and never
    # unsets them: no subprocess inherits them, or a CUDA series would run the Triton backward under a CUDA label
    env = {k: v for k, v in (env or os.environ).items() if k not in ("CQSA_FORWARD", "CQSA_BACKWARD")}
    try:
        with open(path, "a", buffering=1) as f, subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                       text=True, bufsize=1, cwd=cwd or HERE, env=env) as p:
            f.write(f"\n==== {time.strftime('%Y-%m-%d %H:%M:%S')} $ {' '.join(map(str, cmd))}\n")
            for line in p.stdout:
                print(line, end="", flush=True)
                f.write(line)
            rc = p.wait()
    except Exception as e:                                                      # noqa: BLE001
        print(f"!! could not run: {e!r}")
        rc = -1
    print(f"[exit {rc}, {(time.time() - t0) / 60:.1f} min]", flush=True)
    return rc


def bench(args, log):
    return run([PY, "-P", BENCH] + list(args), log, cwd=SRC)


def budget(seconds):
    return [] if (FULL or SMOKE) else ["--budget-s", str(int(seconds * SCALE))]


def _emit(output):                        # the in-process executor (run_cita.py --inproc) collects display outputs here
    sink = globals().get("_cita_outputs")
    if sink is not None:
        sink.append(output)


def show_md(text):
    try:
        get_ipython()                                                           # noqa: F821  (only inside a kernel)
        from IPython.display import display, Markdown
        display(Markdown(text))
    except Exception:                                                           # noqa: BLE001
        print(text)
        _emit({"output_type": "display_data", "data": {"text/markdown": text}, "metadata": {}})


def show_png(path):
    if not os.path.exists(path):
        print(f"(no figure {path})")
        return
    try:
        get_ipython()                                                           # noqa: F821
        from IPython.display import display, Image
        display(Image(filename=path))
    except Exception:                                                           # noqa: BLE001
        print(f"figure: {path}")
        _emit({"output_type": "display_data", "metadata": {},
               "data": {"image/png": base64.b64encode(open(path, "rb").read()).decode("ascii")}})


def results_tarball(name=None):
    """results dir + executed notebook + setup / build / doctor logs -> HERE/<name>; returns (path, size)."""
    host = socket.gethostname().split(".")[0]
    name = name or f"cita_results_{host}_{time.strftime('%Y%m%d_%H%M')}.tar.gz"
    path = os.path.join(HERE, name)
    extras = ["run_cita_executed.ipynb", "setup.log", "build_classic.log", "build_native.log", "doctor.txt", "tiers.txt",
              "tier_a.log", "pip_wheels.log", "pip_src.log", "GIT_COMMIT", "run_cita.log"]
    with tarfile.open(path, "w:gz") as tf:
        tf.add(OUT, arcname=os.path.relpath(OUT, HERE) if OUT.startswith(HERE) else os.path.basename(OUT))
        for e in extras:
            p = os.path.join(HERE, e)
            if os.path.exists(p):
                tf.add(p, arcname=e)
    with open(os.path.join(OUT, "last_tarball.txt"), "w") as f:
        f.write(name + "\n")
    return path, os.path.getsize(path)


print(f"bundle: {HERE}\nresults: {OUT}\nmode: {'SMOKE (plumbing only)' if SMOKE else 'QUICK' if QUICK else 'FULL (no budgets)' if FULL else 'default budgets'}"
      f"  max N {MAX_N}  budget scale {SCALE}  label {LABEL!r}\npython: {PY}")

## 0. Environment (< 1 min)

Host, GPU(s), driver, versions, which `stream_cqsa` modules loaded (`python -m stream_cqsa.doctor`), the git
commit of `src/`, the tiers `setup_cita.sh` reported, and the series this run will use.

In [ ]:
header("0. environment")
try:
    env = dict(hostname=socket.gethostname(), date=time.strftime("%Y-%m-%d %H:%M:%S %Z"), label=LABEL, quick=QUICK, smoke=SMOKE, full=FULL,
               max_n=MAX_N, budget_scale=SCALE, python=sys.version.split()[0], executable=PY, bundle=HERE, out=OUT,
               cuda_visible_devices=os.environ.get("CUDA_VISIBLE_DEVICES"), cpus=os.cpu_count())
    gc = os.path.join(SRC, "GIT_COMMIT")
    env["git_commit"] = open(gc).read().strip() if os.path.exists(gc) else None
    try:
        env["mem_total_gib"] = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1)
    except Exception:                                                           # noqa: BLE001
        env["mem_total_gib"] = None
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=index,name,driver_version,memory.total,memory.used,clocks.max.sm",
                              "--format=csv"], capture_output=True, text=True, timeout=60)
        env["nvidia_smi"] = smi.stdout.strip()
    except Exception as e:                                                      # noqa: BLE001
        env["nvidia_smi"] = f"unavailable: {e!r}"
    # torch / triton / stream_cqsa in a subprocess (this kernel keeps no CUDA context from here)
    probe = r"""
import json, os, sys, torch
d = dict(torch=torch.__version__, cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(), cuda_available=torch.cuda.is_available())
try:
    import triton; d["triton"] = triton.__version__
except Exception as e: d["triton"] = None
import numpy; d["numpy"] = numpy.__version__
d["gpus"] = [dict(index=i, name=p.name, sm=f"{p.major}{p.minor}", mem_gib=round(p.total_memory / 2**30, 1)) for i, p in
             ((i, torch.cuda.get_device_properties(i)) for i in range(torch.cuda.device_count()))]
import stream_cqsa
from stream_cqsa import interface as I
d["stream_cqsa"] = stream_cqsa.__version__; d["stream_cqsa_path"] = os.path.dirname(stream_cqsa.__file__)
d["kernels_available"] = stream_cqsa.kernels_available()
d["cuda_module"] = I.CQSA_CUDA_MODULE if I.cqsa_cuda is not None else None
d["cuda_module_file"] = getattr(I.cqsa_cuda, "__file__", None)
d["cuda_module_import_error"] = repr(I._CQSA_CUDA_IMPORT_ERROR) if I.cqsa_cuda is None and I._CQSA_CUDA_IMPORT_ERROR else None
try:
    from stream_cqsa import native_wave as W
    ext = W.native_ext(); d["native_module"] = ext.__name__ if ext is not None else None; d["native_module_file"] = getattr(ext, "__file__", None)
except Exception as e: d["native_module"] = None; d["native_module_error"] = repr(e)
try:
    import flash_attn; d["flash_attn"] = flash_attn.__version__
except Exception: d["flash_attn"] = None
print("CITA_ENV_JSON " + json.dumps(d))
"""
    r = subprocess.run([PY, "-P", "-c", probe], capture_output=True, text=True, timeout=600, cwd=HERE)
    line = next((ln for ln in r.stdout.splitlines() if ln.startswith("CITA_ENV_JSON ")), None)
    if line is None:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("the environment probe did not report (stream_cqsa / torch not importable?)")
    env.update(json.loads(line[len("CITA_ENV_JSON "):]))
    ks = env["kernels_available"]
    if os.environ.get("CITA_SERIES"):
        SERIES = [s for s in os.environ["CITA_SERIES"].split(",") if s]
    else:
        SERIES = ["classic_Triton", "wave_Triton"] + (["classic_CUDA"] if ks.get("cuda_extension") else []) + (["wave_CUDA"] if ks.get("native_wave") else [])
    BASELINES = ["SDPA"] + (["FA-2"] if env.get("flash_attn") else [])
    env["series"] = SERIES; env["baselines"] = BASELINES
    env["reference_arm"] = "FA-2 (flash_attn)" if env.get("flash_attn") else "SDPA-flash (flash_attn not installed)"
    for k_ in ("hostname", "date", "label", "git_commit", "python", "torch", "cuda", "triton", "numpy", "cpus", "mem_total_gib", "flash_attn",
               "stream_cqsa", "stream_cqsa_path", "cuda_module", "cuda_module_file", "cuda_module_import_error", "native_module", "native_module_file",
               "kernels_available", "series", "baselines", "reference_arm", "cuda_visible_devices"):
        print(f"{k_:26s} {env.get(k_)}")
    print("gpus:")
    for g in env["gpus"]:
        print(f"  {g}")
    print("nvidia-smi:\n  " + env["nvidia_smi"].replace("\n", "\n  "))
    for f in ("tiers.txt", "doctor.txt"):
        p = os.path.join(HERE, f)
        if os.path.exists(p):
            print(f"\n--- {f} (setup_cita.sh) ---\n" + open(p).read().rstrip())
    print("\n--- python -m stream_cqsa.doctor (now) ---")
    rc = run([PY, "-P", "-m", "stream_cqsa.doctor"], "doctor.log", cwd=HERE)
    env["doctor_rc"] = rc
    with open(os.path.join(OUT, "env.json"), "w") as f:
        json.dump(env, f, indent=1)
    record("env", True, f"series {SERIES}, baselines {BASELINES}, reference {env['reference_arm']}")
except Exception:                                                               # noqa: BLE001
    traceback.print_exc()
    record("env", False, "see traceback")
print(f"\nseries: {SERIES}   baselines: {BASELINES}")

## 1a. Correctness: the test suite (~2-5 min)

`pytest -q` on `tests/test_v3_quorum_sizes.py` (every quorum set `c` x N in {64K, 128K} x itr in {1, 2},
forward and backward against SDPA-flash, the decomposition asserted; the v2.2.3 c=73 regression),
`tests/test_api.py`, `tests/test_evict_inputs.py`, `tests/test_ultimate_routing.py`. Tests that need the wave
CUDA module skip without it. Note: `test_v3_quorum_sizes` asks for `kernel="cuda"`; on a Triton-only install the
classic engine runs its Triton inner kernel there (the forward info still says `classic`), so it is the Triton
path that is being tested in that case. Output: `results/.../logs/pytest.log`.

In [ ]:
header("1a. pytest")
try:
    tests = [os.path.join(SRC, "tests", t) for t in ("test_v3_quorum_sizes.py", "test_api.py", "test_evict_inputs.py", "test_ultimate_routing.py")]
    rc = run([PY, "-P", "-m", "pytest", "-q", "-p", "no:cacheprovider", "-rs", "--no-header", "--color=no"] + tests, "pytest.log", cwd=SRC)
    summary = ""
    try:
        lines = [ln for ln in open(os.path.join(OUT, "logs", "pytest.log")).read().splitlines() if " passed" in ln or " failed" in ln or " error" in ln]
        summary = lines[-1].strip("= ") if lines else ""
    except Exception:                                                           # noqa: BLE001
        pass
    record("pytest", rc == 0, summary or f"exit {rc}")
except Exception:                                                               # noqa: BLE001
    traceback.print_exc()
    record("pytest", False, "see traceback")

## 1b. Correctness inline: `attention()` vs SDPA-flash, `itr=1` pinned (~1-2 min)

For every available series, N in {64K, 256K} (SMOKE: 16K / 32K), fp16, B=1 H=8 D=64, causal: a forward
(`stream_cqsa.attention(..., kernel=<series>, itr=1, c=7, return_info=True)`, the info must report 7 subproblems)
and a forward + backward (`bwd_itr=1`, gradients through `torch.autograd.grad`) against
`F.scaled_dot_product_attention` on the flash backend with the same inputs; `tests/_decomp.py:assert_decomposed`
checks the info of every call (engine classic / wave, itr 1, c 7). The wave series' backward runs on the classic
engine when the wave CUDA module is absent (documented routing), which the engine column shows. Tolerances: output 1e-3, gradients 1.5e-3
(norm-relative; the A100 values are 1-3e-4 / 3-4e-4). This cell runs in the notebook's own process and frees the
GPU afterwards.

In [ ]:
header("1b. inline correctness vs SDPA-flash")
rows = []
try:
    import torch
    import torch.nn.functional as F
    from torch.nn.attention import sdpa_kernel, SDPBackend
    sys.path.insert(0, os.path.join(SRC, "tests"))
    from _decomp import assert_decomposed
    import stream_cqsa
    from stream_cqsa.autoconfig import QUORUM_SETS
    KERNEL_OF = {"classic_CUDA": "cuda", "classic_Triton": "triton", "wave_CUDA": "wave-cuda", "wave_Triton": "wave-triton"}
    TOL_OUT, TOL_GRAD = 1e-3, 1.5e-3
    B_, H_, D_, c = 1, 8, 64, 7
    Ns = [16 * K, 32 * K] if SMOKE else [64 * K, 256 * K]

    def rel(a, b):
        return ((a.double() - b.double()).norm() / b.double().norm()).item()

    for N in Ns:
        g = torch.Generator(device="cuda").manual_seed(N)
        q, k, v, dout = (torch.randn(B_, H_, N, D_, device="cuda", dtype=torch.float16, generator=g) for _ in range(4))
        qr, kr, vr = (t.clone().requires_grad_(True) for t in (q, k, v))
        with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
            ref = F.scaled_dot_product_attention(qr, kr, vr, is_causal=True)
        ref_g = [x.detach() for x in torch.autograd.grad(ref, (qr, kr, vr), dout)]
        ref = ref.detach()
        del qr, kr, vr
        for s in SERIES:
            for direction in ("fwd", "fwd+bwd"):
                row = dict(series=s, N=N, direction=direction, kernel_requested=KERNEL_OF.get(s, s))
                t0 = time.time()
                # attention(kernel="triton") leaves CQSA_FORWARD / CQSA_BACKWARD=triton in this process: clear them before every
                # call so a CUDA series after a Triton one runs its CUDA kernels (tests/test_api.py does the same)
                os.environ.pop("CQSA_FORWARD", None); os.environ.pop("CQSA_BACKWARD", None)
                try:
                    pins = dict(is_causal=True, kernel=KERNEL_OF[s], itr=1, c=c, interest_set=QUORUM_SETS[c], return_info=True)
                    if direction == "fwd":
                        with torch.no_grad():
                            out, info = stream_cqsa.attention(q, k, v, **pins)
                        assert_decomposed(info, itr=1, c=c, n_subproblems=c)       # the forward info names the subproblem count
                        grads = None
                    else:
                        qg, kg, vg = (t.clone().requires_grad_(True) for t in (q, k, v))
                        out, info = stream_cqsa.attention(qg, kg, vg, bwd_itr=1, **pins)
                        assert_decomposed(info, itr=1, c=c)                        # the grad-path info: engine / kernel / itr / c
                        grads = torch.autograd.grad(out, (qg, kg, vg), dout)
                        del qg, kg, vg
                    from stream_cqsa import interface as _I
                    row.update(engine=info.get("engine"), kernel=info.get("kernel"), itr=info.get("itr"), n_subproblems=info.get("n_subproblems"),
                               backward_kernel=(_I.backward_kernel() if direction != "fwd" else None),
                               rel_out=rel(out, ref), wall_s=round(time.time() - t0, 2))
                    if grads is not None:
                        row.update(rel_grads=max(rel(a, b) for a, b in zip(grads, ref_g)), rel_grads_each=[rel(a, b) for a, b in zip(grads, ref_g)])
                    row["ok"] = row["rel_out"] < TOL_OUT and (grads is None or row["rel_grads"] < TOL_GRAD)
                    row["error"] = None if row["ok"] else f"above tolerance (out {TOL_OUT}, grads {TOL_GRAD})"
                    del out, grads
                except Exception as e:                                          # noqa: BLE001
                    row.update(ok=False, error=f"{type(e).__name__}: {str(e)[:300]}")
                    traceback.print_exc()
                rows.append(row)
                torch.cuda.synchronize(); torch.cuda.empty_cache()
        del q, k, v, dout, ref, ref_g
        torch.cuda.empty_cache()
    hdr = f"{'series':15s} {'N':>8s} {'dir':8s} {'engine/kernel (bwd)':24s} {'itr':>3s} {'n_sub':>5s} {'rel out':>9s} {'rel grads':>9s} {'s':>6s}  ok"
    print(hdr); print("-" * len(hdr))
    for r in rows:
        ek = f"{r.get('engine')}/{r.get('kernel')}" + (f" (bwd {r['backward_kernel']})" if r.get("backward_kernel") else "")
        rg = f"{r['rel_grads']:9.1e}" if r.get("rel_grads") is not None else f"{'-':>9s}"
        print(f"{r['series']:15s} {r['N']:8d} {r['direction']:8s} {ek:24s} {str(r.get('itr', '-')):>3s} {str(r.get('n_subproblems', '-')):>5s} "
              f"{r.get('rel_out', float('nan')):9.1e} {rg} {r.get('wall_s', 0):6.2f}  {'ok' if r['ok'] else 'FAIL ' + str(r.get('error'))}")
    with open(os.path.join(OUT, "inline_correctness.json"), "w") as f:
        json.dump(dict(reference="SDPA-flash (torch.nn.functional.scaled_dot_product_attention, SDPBackend.FLASH_ATTENTION)", tol_out=TOL_OUT,
                       tol_grads=TOL_GRAD, B=B_, H=H_, D=D_, c=c, rows=rows), f, indent=1)
    record("inline_correctness", bool(rows) and all(r["ok"] for r in rows), f"{sum(r['ok'] for r in rows)}/{len(rows)} rows within tolerance")
except Exception:                                                               # noqa: BLE001
    traceback.print_exc()
    record("inline_correctness", False, "see traceback")
finally:
    os.environ.pop("CQSA_FORWARD", None); os.environ.pop("CQSA_BACKWARD", None)
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:                                                           # noqa: BLE001
        pass

## 2. E0: correctness experiment (A100: 21 min for 128 rows; budget 30 min; quick: 48 rows, 15 min)

`bench.py run --exp E0`: every series x c in {3, 7, 31, 133} x itr in {1, 2} x accumulator {gpu, cpu} x
{fwd, fwd+bwd} at N = 256K fp16 D=64, plus bf16 and D=128 rows at c=7; output and gradients against the
reference arm (FA-2, or SDPA-flash without `flash_attn`) and against an fp64 reference on 128 sampled rows,
with the ratio to the reference arm's own fp64 error (PLAN.md asks <= 1.2). `CITA_QUICK=1`: c in {7, 31}, itr 1.

In [ ]:
header("2. E0 correctness", "quick grid" if QUICK else "smoke" if SMOKE else "full grid")
try:
    args = ["run", "--exp", "E0", "--out", OUT, "--series", ",".join(SERIES), "--ref", "fa2"]
    if SMOKE:
        args += ["--smoke"]
    elif QUICK:
        args += ["--c", "7,31", "--itr", "1"]
    args += budget(900 if QUICK else 1800)
    rc = bench(args, "E0.log")
    record("E0", rc == 0, f"exit {rc}")
except Exception:                                                               # noqa: BLE001
    traceback.print_exc()
    record("E0", False, "see traceback")

## 3. E1 forward: wall and peak memory vs N (A100: 32 min at reps 1; budget 30 min)

N in {128K, 256K, 512K, 1M, 2M, 4M} (up to `CITA_MAX_N`; quick: up to 1M), c=7, the depth rule of PLAN.md
(the smallest itr >= 1 whose plan fits the device), device-resident Q/K/V, fp16 D=64; every series plus the
monolithic baselines (SDPA; FA-2 when installed); 1 warm-up + 2 timed runs, the fp64 spot check.

In [ ]:
header("3. E1 forward")
try:
    Ns = [n for n in N_LADDER if n <= min(MAX_N, M if QUICK else 4 * M)]
    args = ["run", "--exp", "E1", "--part", "fwd", "--out", OUT, "--series", ",".join(SERIES + BASELINES), "--reps", "2"]
    args += ["--smoke"] if SMOKE else ["--N", ",".join(map(str, Ns))]
    args += budget(1800)
    rc = bench(args, "E1_fwd.log")
    record("E1_fwd", rc == 0, f"exit {rc}, " + ("smoke N" if SMOKE else f"N up to {Ns[-1]}"))
except Exception:                                                               # noqa: BLE001
    traceback.print_exc()
    record("E1_fwd", False, "see traceback")

## 4. E1 forward+backward: wall and peak memory vs N (A100: not measured; ~2.5 h estimated; budget 30 min)

The same ladder with `torch.autograd.grad` of the output (one forward + backward timed, the backward's depth
pinned to the forward's). Skipped under `CITA_QUICK=1`. The budget stops at the largest N that fits 30 min;
re-run the notebook (or `CITA_BUDGET_SCALE=4`) for the rest.

In [ ]:
header("4. E1 forward+backward")
try:
    if QUICK:
        print("skipped under CITA_QUICK=1")
        record("E1_bwd", True, "skipped (quick)")
    else:
        Ns = [n for n in N_LADDER if n <= MAX_N]
        args = ["run", "--exp", "E1", "--part", "bwd", "--out", OUT, "--series", ",".join(SERIES + BASELINES), "--reps", "2"]
        args += ["--smoke"] if SMOKE else ["--N", ",".join(map(str, Ns))]
        args += budget(1800)
        rc = bench(args, "E1_bwd.log")
        record("E1_bwd", rc == 0, f"exit {rc}, " + ("smoke N" if SMOKE else f"N up to {Ns[-1]}"))
except Exception:                                                               # noqa: BLE001
    traceback.print_exc()
    record("E1_bwd", False, "see traceback")

## 5. E3 forward: the quorum size c at N = 1M (A100: 25 min at reps 3; budget 15 min)

c in {3, 7, 13, 21, 31, 57, 73, 91, 133}, itr 1, device-resident, every series; 1 warm-up + 2 timed runs,
errors against the reference arm. Skipped under `CITA_QUICK=1`.

In [ ]:
header("5. E3 forward (c sweep at 1M)")
try:
    if QUICK:
        print("skipped under CITA_QUICK=1")
        record("E3_fwd", True, "skipped (quick)")
    else:
        args = ["run", "--exp", "E3", "--part", "fwd", "--out", OUT, "--series", ",".join(SERIES), "--reps", "2"]
        args += ["--smoke"] if SMOKE else []
        args += budget(900)
        rc = bench(args, "E3_fwd.log")
        record("E3_fwd", rc == 0, f"exit {rc}")
except Exception:                                                               # noqa: BLE001
    traceback.print_exc()
    record("E3_fwd", False, "see traceback")

## 6. E4: depth x accumulator placement at N = 1M (A100: 45 min at reps 3 for 24 of 32 rows; budget 15 min)

itr in {1, 2} x accumulator {gpu, cpu}, c=7, device-resident Q/K/V, forward and forward+backward, every series;
1 warm-up + 2 timed runs. Skipped under `CITA_QUICK=1`.

In [ ]:
header("6. E4 (itr x acc at 1M)")
try:
    if QUICK:
        print("skipped under CITA_QUICK=1")
        record("E4", True, "skipped (quick)")
    else:
        args = ["run", "--exp", "E4", "--out", OUT, "--series", ",".join(SERIES), "--reps", "2"]
        args += ["--smoke"] if SMOKE else []
        args += budget(900)
        rc = bench(args, "E4.log")
        record("E4", rc == 0, f"exit {rc}")
except Exception:                                                               # noqa: BLE001
    traceback.print_exc()
    record("E4", False, "see traceback")

## 7. E6: precision and head dim at N = 1M (A100: not measured, ~60-75 min estimated; budget 15 min)

{fp16, bf16} x D in {64, 128}, c=7 itr=1, forward and forward+backward, every series plus the baselines;
1 warm-up + 2 timed runs. Skipped under `CITA_QUICK=1`.

In [ ]:
header("7. E6 (bf16 and D=128 at 1M)")
try:
    if QUICK:
        print("skipped under CITA_QUICK=1")
        record("E6", True, "skipped (quick)")
    else:
        args = ["run", "--exp", "E6", "--out", OUT, "--series", ",".join(SERIES + BASELINES), "--reps", "2"]
        args += ["--smoke"] if SMOKE else []
        args += budget(900)
        rc = bench(args, "E6.log")
        record("E6", rc == 0, f"exit {rc}")
except Exception:                                                               # noqa: BLE001
    traceback.print_exc()
    record("E6", False, "see traceback")

## 8. Report (< 1 min)

`bench.py report --out <results>`: `report.md` (one table per experiment, the worst-case accuracy table of E0,
the quadratic / linear fits of E1), `fits.json`, `STATUS.md` (rows done / oom / error / missing per experiment)
and the figures `E0.png`, `E1.png`, `E3.png`, `E4.png`, `E6.png`, shown inline below. The `/FA-2` ratio column
is empty when `flash_attn` is not installed; `/SDPA` is the ratio to torch's flash backend. `STATUS.md` counts the
rows of the full plan (every series, both baselines, E2 and E5 too), so its "missing" column includes the series
this install does not have, the FA-2 rows without `flash_attn`, E2 / E5 (not run by this notebook) and the rows a
budget deferred.

In [ ]:
header("8. report")
try:
    rc = bench(["report", "--out", OUT], "report.log")
    rep = os.path.join(OUT, "report.md")
    if os.path.exists(rep):
        text = open(rep).read()
        show_md("\n".join(ln for ln in text.splitlines() if not ln.startswith("![")))
    for exp in ("E0", "E1", "E3", "E4", "E6"):
        p = os.path.join(OUT, f"{exp}.png")
        if os.path.exists(p):
            print(f"\n{exp}.png")
            show_png(p)
    st = os.path.join(OUT, "STATUS.md")
    if os.path.exists(st):
        print("\n" + open(st).read())
    record("report", rc == 0 and os.path.exists(rep), f"exit {rc}")
except Exception:                                                               # noqa: BLE001
    traceback.print_exc()
    record("report", False, "see traceback")

## 9. Results tarball

`cita_results_<host>_<date>.tar.gz` in the bundle directory: the results directory (`results.jsonl`,
`report.md`, `STATUS.md`, `fits.json`, `E*.png`, `env.json`, `inline_correctness.json`, `cell_status.json`,
`logs/`), the executed notebook as saved so far, `setup.log`, `build_classic.log`, `build_native.log`,
`doctor.txt`, `tiers.txt`. `run_cita.py` rebuilds the same tarball once more after the final save of the
notebook, so the copy to bring back is the one printed at the end of `python run_cita.py`.

In [ ]:
header("9. results tarball")
try:
    print(f"{'cell':22s} {'ok':4s} at(min)  note")
    for name, s in STATUS.items():
        print(f"{name:22s} {'ok' if s['ok'] else 'FAIL':4s} {s['at_min']:7.1f}  {s['note']}")
    path, size = results_tarball()
    print(f"\nresults tarball: {path}  ({size / 2**20:.1f} MiB)")
    print(f"total wall: {(time.time() - T_START) / 60:.1f} min")
    record("tarball", True, os.path.basename(path))
except Exception:                                                               # noqa: BLE001
    traceback.print_exc()
    record("tarball", False, "see traceback")